# Simple ZScore

The logic is:

High mvrv_zscore (overvalued, e.g. +3) → preference = −3 → buy less
Low mvrv_zscore (undervalued, e.g. −2) → preference = +2 → buy more
If mvrv_zscore column is missing (e.g. custom data without MVRV), falls back to np.zeros → uniform buying

#### Comparison to MVRV strategy:

| "" |SimpleZScoreStrategy|MVRVStrategy|
|---|---|---|
|Preference formula|−mvrv_zscore (one line)|Weighted blend of 3 signals (70/20/10)|
|Signals used|1 (mvrv_zscore)|3 (mvrv_zscore, mvrv_percentile, price_vs_ma)|
|Modulation|None|+ gradient, acceleration, volatility dampening|
|Purpose|Teaching example / baseline|Production strategy|

**example:**
&ensp;Aug 2, 2016 — price = $542.31, zscore = 0.1 <br>This day is somewhere in the middle of a 365-day window. Let's say it's day index d=180 (day 181 of 365).

**Step A — preference:**
preference = −0.1 <br>(zscore = 0.1 means Bitcoin is slightly overvalued -> mild signal to buy less)

**Step B — raw:**
$raw_{180} = \frac{1}{365} \exp{-0.1}$ = 0.002740×0.9048 = 0.002479 <br>For comparison, a perfectly neutral day (zscore = 0) would have:
$raw_{neutral} = \frac{1}{365} \exp{0}$ <br>= 0.002740

**Step C — stable_signal:**
Suppose the preceding 180 days had an average zscore of +0.8 (overvalued period). Then their raw values averaged around: <br>running_mean ≈ $\frac{1}{365} \exp{-0.8}$ ≈ 0.002740×0.449 = 0.001230
Then: <br>$signal_{180} = \frac{0.002479}{0.001230}$ ≈ 2.016 <br>Even though zscore = 0.1 means slightly overvalued, relative to the preceding very-overvalued days, this day looks like a buying opportunity.

**Step D — proposed:**
$proposed_{180} = 2.016 × \frac{1}{365}$ = 0.005523

**Step E — clipped:**
With 184 days remaining after this day, budget constraints give roughly: <br>max_upper = min(0.1, remaining − 184×0.00001) ≈ 0.1 <br>min_lower = max(0.00001, remaining − 184×0.1) <br>The proposed 0.005523 is within bounds → weight = 0.0055 (approximately)

In [1]:
import sys
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

from stacksats.model_development import precompute_features
from stacksats.runner import StrategyRunner
from stacksats.strategy_types import ExportConfig
from stacksats.strategies.stable.signals.simple_zscore import SimpleZScoreStrategy
from stacksats.strategies.stable.baselines.uniform import UniformStrategy

_root = Path.cwd()
while not (_root / "src").exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from src import config as src_config, data_utils, plots, strategy_utils

STACKSATS_DATA_PATH = src_config.STACKSATS_DATA_PATH
RAW_PATH = src_config.RAW_PATH

data_utils.check_stacksats_data(STACKSATS_DATA_PATH, RAW_PATH)
# Load raw BTC data
btc_df = pl.read_parquet(STACKSATS_DATA_PATH).sort("date")
btc_df = btc_df.with_columns(pl.col("date").cast(pl.Datetime))


In [2]:
btc_train = (
    btc_df
    .filter(
        (pl.col("date") >= pl.datetime(2010, 8, 16)) &
        (pl.col("date") <= pl.datetime(2023, 12, 31)) &
        pl.col("price_usd").is_not_null()
    )
    .sort("date")
)

print("Filtered rows:", btc_train.height)
print(
    btc_train.select(
        pl.col("date").min().alias("min_date"),
        pl.col("date").max().alias("max_date")
    )
)

Filtered rows: 4886
shape: (1, 2)
┌─────────────────────┬─────────────────────┐
│ min_date            ┆ max_date            │
│ ---                 ┆ ---                 │
│ datetime[μs]        ┆ datetime[μs]        │
╞═════════════════════╪═════════════════════╡
│ 2010-08-16 00:00:00 ┆ 2023-12-31 00:00:00 │
└─────────────────────┴─────────────────────┘


In [3]:
# Strategy runner
runner = StrategyRunner()

cycle_results = {}

for cycle in plots.calendar_cycles:
    result = strategy_utils.process_cycle_year_by_year(
        cycle=cycle,
        btc_data=btc_train,
        runner=runner,
        dynamic_strategy=SimpleZScoreStrategy(),
        total_budget_usd=src_config.TOTAL_BUDGET_USD,
        top_buy_quantile=src_config.TOP_BUY_QUANTILE
    )

    cycle_results[cycle["label"]] = result


Processing Cycle 1: 2010-2013
shape: (1, 3)
┌─────────────────────┬─────────────────────┬──────┐
│ min_date            ┆ max_date            ┆ rows │
│ ---                 ┆ ---                 ┆ ---  │
│ datetime[μs]        ┆ datetime[μs]        ┆ u32  │
╞═════════════════════╪═════════════════════╪══════╡
│ 2010-08-16 00:00:00 ┆ 2013-12-31 00:00:00 ┆ 1234 │
└─────────────────────┴─────────────────────┴──────┘
2010: skipped, less than 365 rows
2010: skipped, less than 365 rows
2011: exported 365 rows


2011: exported 365 rows
2012: exported 365 rows
2012: exported 365 rows


2013: exported 365 rows
2013: exported 365 rows

Processing Cycle 2: 2014-2017
shape: (1, 3)
┌─────────────────────┬─────────────────────┬──────┐
│ min_date            ┆ max_date            ┆ rows │
│ ---                 ┆ ---                 ┆ ---  │
│ datetime[μs]        ┆ datetime[μs]        ┆ u32  │
╞═════════════════════╪═════════════════════╪══════╡
│ 2014-01-01 00:00:00 ┆ 2017-12-31 00:00:00 ┆ 1461 │
└─────────────────────┴─────────────────────┴──────┘
2014: exported 365 rows
2014: exported 365 rows


2015: exported 365 rows
2015: exported 365 rows
2016: exported 365 rows


2016: exported 365 rows
2017: exported 365 rows
2017: exported 365 rows

Processing Cycle 3: 2018-2021
shape: (1, 3)
┌─────────────────────┬─────────────────────┬──────┐
│ min_date            ┆ max_date            ┆ rows │
│ ---                 ┆ ---                 ┆ ---  │
│ datetime[μs]        ┆ datetime[μs]        ┆ u32  │
╞═════════════════════╪═════════════════════╪══════╡
│ 2018-01-01 00:00:00 ┆ 2021-12-31 00:00:00 ┆ 1461 │
└─────────────────────┴─────────────────────┴──────┘
2018: exported 365 rows


2018: exported 365 rows
2019: exported 365 rows
2019: exported 365 rows


2020: exported 365 rows
2020: exported 365 rows
2021: exported 365 rows


2021: exported 365 rows

Processing Cycle 4: 2022-2023
shape: (1, 3)
┌─────────────────────┬─────────────────────┬──────┐
│ min_date            ┆ max_date            ┆ rows │
│ ---                 ┆ ---                 ┆ ---  │
│ datetime[μs]        ┆ datetime[μs]        ┆ u32  │
╞═════════════════════╪═════════════════════╪══════╡
│ 2022-01-01 00:00:00 ┆ 2023-12-31 00:00:00 ┆ 730  │
└─────────────────────┴─────────────────────┴──────┘
2022: exported 365 rows
2022: exported 365 rows
2023: exported 365 rows


2023: exported 365 rows


In [4]:
cols = plots.StrategyColumns(
    weight="dynamic_weight",
    spd="sats_per_dollar_dynamic",
    sats_accum="sats_accum_dynamic",
)

# Combine all 4 cycles into one DataFrame
combined_plot_df = pd.concat(
    [r["plot_df"] for r in cycle_results.values()]
).sort_values("date").reset_index(drop=True)

# per cycle plots
cycle_plots = plots.plot_strategy_by_cycle(combined_plot_df, cols, "Simple Z-Score")
plt.show()

In [5]:
# per year plots
year_plots = plots.plot_strategy_by_year(combined_plot_df, cols, "Simple Z-Score")
plt.show()

In [6]:
strategies_train = {
    "dynamic": SimpleZScoreStrategy(),
    "baseline": UniformStrategy(),
}

train_years  = range(src_config.TRAIN_START_YEAR, src_config.SPLIT_YEAR)   # 2018–2023
merged_train = strategy_utils.run_year_by_year(strategies_train, btc_df, train_years, runner)

perf_vf   = strategy_utils.compute_performance_summary(merged_train, "dynamic", "baseline")

print(f"[Train] Simple Z-Score SPD : {perf_vf['sats_per_dollar_dynamic']:.2f}")
print(f"[Train] Baseline SPD       : {perf_vf['sats_per_dollar_baseline']:.2f}")
print(f"Simple Z-Score {abs(perf_vf['pct_diff_vs_baseline']):.2f}% {perf_vf['performance_label']} than Uniform")


2018: exported 365 rows
2018: exported 365 rows
2019: exported 365 rows


2019: exported 365 rows
2020: exported 365 rows


2020: exported 365 rows
2021: exported 365 rows
2021: exported 365 rows
2022: exported 365 rows


2022: exported 365 rows
2023: exported 365 rows
2023: exported 365 rows
[Train] Simple Z-Score SPD : 8072.58
[Train] Baseline SPD       : 8418.79
Simple Z-Score 4.11% worse than Uniform


## Adding Test data

In [7]:
# Testing window: 2024 onwards (held-out period)
btc_df_test = (
    btc_df
    .filter(
        (pl.col("date") >= pl.datetime(2024, 1, 1)) &
        pl.col("price_usd").is_not_null()
    )
    .sort("date")
)

print("Test rows:", btc_df_test.height)
print(
    btc_df_test.select(
        pl.col("date").min().alias("min_date"),
        pl.col("date").max().alias("max_date")
    )
)

Test rows: 803
shape: (1, 2)
┌─────────────────────┬─────────────────────┐
│ min_date            ┆ max_date            │
│ ---                 ┆ ---                 │
│ datetime[μs]        ┆ datetime[μs]        │
╞═════════════════════╪═════════════════════╡
│ 2024-01-01 00:00:00 ┆ 2026-03-13 00:00:00 │
└─────────────────────┴─────────────────────┘


In [8]:
strategies_test = {
    "dynamic": SimpleZScoreStrategy(),
    "baseline": UniformStrategy(),
}

test_years  = range(src_config.SPLIT_YEAR, src_config.TEST_END_YEAR + 1)   # 2024–2025
merged_test = strategy_utils.run_year_by_year(strategies_test, btc_df_test, test_years, runner)

perf_test_vf   = strategy_utils.compute_performance_summary(merged_test, "dynamic", "baseline")


2024: exported 365 rows


2024: exported 365 rows
2025: exported 365 rows


2025: exported 365 rows


In [9]:
combined_plot_df = (
    pd.concat([merged_train.to_pandas(), merged_test.to_pandas()])
    .sort_values("date")
    .reset_index(drop=True)
)
combined_plot_df["date"] = pd.to_datetime(combined_plot_df["date"])

cols = plots.StrategyColumns(
    weight="dynamic_weight",
    spd="sats_per_dollar_dynamic",
    sats_accum="sats_accum_dynamic",
)

# Full period
full_plot = plots.plot_strategy_full_period(
    combined_plot_df, 
    cols, 
    "Simple Z-Score", 
    date_range=("2018-01-01", str(combined_plot_df["date"].max().date())),
    test_start_date="2024-01-01"
)
plt.show()

## Full Window: 2010–2023 - Not to be used

A single chart spanning the entire date range with Bitcoin halving cycle bands shaded in the background.  Heavy buy days (top 10% of SimpleZScore allocation weights globally) are marked as red circles.

In [10]:
# Precompute MVRV z-score features for hover tooltips
features_df = precompute_features(btc_df).select(["date", "mvrv_zscore"])

def _dedup(batch, weight_col: str) -> pl.DataFrame:
    """Flatten overlapping rolling windows to one row per date.

    When a strategy runs over a multi-year range the runner produces
    overlapping rolling 365-day windows internally.  Each window assigns
    its own weight to every date it covers, so a single date may appear in
    dozens of windows with slightly different weights.  _dedup keeps the
    first weight seen for each date.
    """
    return (
        batch.to_dataframe()
        .group_by("date")
        .agg(
            pl.first("weight").alias(weight_col),
            pl.first("price_usd").alias("price_usd"),
        )
        .sort("date")
    )

In [11]:
WINDOW_START = "2010-01-01"
WINDOW_END   = "2023-12-31"
config = ExportConfig(range_start=WINDOW_START, range_end=WINDOW_END)

zscore_dd  = _dedup(runner.export(SimpleZScoreStrategy(), config, btc_df=btc_df), "zscore_weight")
uniform_dd = (
    _dedup(runner.export(UniformStrategy(), config, btc_df=btc_df), "baseline_weight")
    .select(["date", "baseline_weight"])
)

merged_full = zscore_dd.join(uniform_dd, on="date", how="inner")

zscore_w_sum    = merged_full["zscore_weight"].sum()
baseline_w_sum  = merged_full["baseline_weight"].sum()

merged_full = merged_full.with_columns([
    (1e8 / pl.col("price_usd")).alias("sats_per_dollar_dynamic"),
    (1e8 / pl.col("price_usd")).alias("sats_per_dollar_baseline"),
    ((pl.col("zscore_weight")   / zscore_w_sum   * src_config.TOTAL_BUDGET_USD) / pl.col("price_usd") * 1e8).alias("sats_accum_dynamic"),
    ((pl.col("baseline_weight") / baseline_w_sum * src_config.TOTAL_BUDGET_USD) / pl.col("price_usd") * 1e8).alias("sats_accum_baseline"),
])


zscore_spd  = (merged_full["zscore_weight"]  * merged_full["sats_per_dollar_dynamic"]).sum() / merged_full["zscore_weight"].sum()
baseline_spd = (merged_full["baseline_weight"] * merged_full["sats_per_dollar_baseline"]).sum() / merged_full["baseline_weight"].sum()
excess_pct  = (zscore_spd - baseline_spd) / baseline_spd * 100

threshold = merged_full["zscore_weight"].quantile(src_config.TOP_BUY_QUANTILE)
merged_full = merged_full.join(features_df, on="date", how="left").with_columns(
    (pl.col("zscore_weight") >= threshold).alias("heavy_buy")
)
df = merged_full.to_pandas()

cols = plots.StrategyColumns(
    weight="zscore_weight",
    spd="sats_per_dollar_dynamic",
    sats_accum="sats_accum_dynamic",
)

# Full period
full_plot = plots.plot_strategy_full_period(df, cols, "Simple Z-Score", ("2010-08-16", "2023-12-31"))
plt.show()
